# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()

print("Dataset Name:", metadata.get('name'))
print("Description:", metadata.get('description'))
print("Published:", metadata.get('datePublished'))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, a *record set* is a logical table of data, often mapping to a CSV file or similar tabular resource.

Let's enumerate all available record sets in the dataset by their `@id`, along with their column/field `@id`s.

In [ ]:
# List all record sets and their fields by @id
print("Record Sets in this Croissant package:")
recsets = list(dataset.metadata.record_sets)
for rset in recsets:
    print(f"- RecordSet @id: {rset.id} | name: {getattr(rset, 'name', 'N/A')}")
    print("  Fields (columns):")
    for field in rset.fields:
        print(f"    - {field.id} ({getattr(field, 'name', 'N/A')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 

**NOTE:** For this notebook, we will select the first record set found. You may adjust the selected record set and fields according to your research needs.

All record set and field references use their `@id` as per the Croissant specification.

In [ ]:
# Select record set and load all records as a DataFrame
record_sets = [rs.id for rs in dataset.metadata.record_sets]

dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set: {record_set_id} - shape: {df.shape}")

# For this notebook, use the first record set
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id is not None:
    print("Columns available in main DataFrame:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Select a numeric field (e.g., 'cr:coverage_percent' or 'cr:molecular_weight') for basic analysis.

You should replace `<numeric_field_id>` and `<group_field_id>` with actual field `@id`s available for your record set, as printed in the previous step.

In [ ]:
import numpy as np
# Pick numeric field @id (example: 'cr:coverage_percent') and a group-by field (example: 'cr:accession')
# If not sure, copy-paste an actual field @id from the previous overview cell

numeric_field = None
group_field = None
if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Find the first numeric column (float or int) in the DataFrame
    for col in df.columns:
        # Heuristic: Try to cast first non-object column to numeric
        try:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
            df[col].astype(float)
            numeric_field = col
            break
        except:
            continue
    # For group by, use the first available non-numeric column as an example
    for col in df.columns:
        if col != numeric_field and pd.api.types.is_object_dtype(df[col]):
            group_field = col
            break
else:
    print("No data to analyze.")

print(f"Selected numeric_field: {numeric_field}")
print(f"Selected group_field: {group_field}")

if numeric_field and main_record_set_id:
    df = dataframes[main_record_set_id]
    # Convert field if necessary
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = np.nanpercentile(df[numeric_field], 50)  # median as an example
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    filtered_df[f'{numeric_field}_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())

    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and main_record_set_id:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.xlabel(numeric_field)
    plt.title(f'Distribution of {numeric_field}')
    plt.show()
    
    if group_field in df.columns:
        plt.figure(figsize=(10, 5))
        # Show top 10 groups
        order = df[group_field].value_counts().index[:10]
        sns.boxplot(x=group_field, y=numeric_field, data=df[df[group_field].isin(order)], order=order)
        plt.title(f'{numeric_field} by {group_field} (Top 10)')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and process the FAIR\u2072 dataset using the `mlcroissant` library. This included listing all record sets and their fields by `@id`, extracting tabular data, analyzing a numeric field, and visualizing value distributions. You can extend this notebook to perform more specialized analyses by inspecting other fields and record sets using their `@id`, as defined in the dataset's Croissant JSON-LD schema.